%md
# Capture Inference Payloads to Delta Table

This section demonstrates how to capture all request payloads and responses from the model serving endpoint and store them in a Delta table for monitoring and analysis.

In [0]:
# Setup - Load your data
CATALOG_NAME = "ml_catalog"
SCHEMA_NAME = "titanic_schema"
TRAIN_TABLE_NAME = "train"


train = spark.table(f"{CATALOG_NAME}.{SCHEMA_NAME}.{TRAIN_TABLE_NAME}").toPandas()
features = ["Fare", "Age", "Pclass", "SibSp", "Parch", "Sex"] 
X = train[features]
y = train["Survived"]

# Let's only test using 5 samples
X = X.head()
y = y.head()

In [0]:
train.head(10)

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, TimestampType, DoubleType, IntegerType
from datetime import datetime

# Define the table to store inference logs
CATALOG_NAME = "ml_catalog"
SCHEMA_NAME = "titanic_schema"
INFERENCE_LOG_TABLE = f"{CATALOG_NAME}.{SCHEMA_NAME}.inference_logs"

# Create the Delta table schema
schema = StructType([
    StructField("request_id", StringType(), False),
    StructField("timestamp", TimestampType(), False),
    StructField("endpoint_name", StringType(), False),
    StructField("model_id", StringType(), True),
    StructField("model_version", StringType(), True),
    StructField("input_fare", DoubleType(), True),
    StructField("input_age", DoubleType(), True),
    StructField("input_pclass", IntegerType(), True),
    StructField("input_sibsp", IntegerType(), True),
    StructField("input_parch", IntegerType(), True),
    StructField("input_sex", StringType(), True),
    StructField("prediction", IntegerType(), True),
    StructField("prediction_probability", DoubleType(), True),
    StructField("actual", IntegerType(), True),
    StructField("response_time_ms", DoubleType(), True),
    StructField("status_code", IntegerType(), True)
])

# Create the table if it doesn't exist
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {INFERENCE_LOG_TABLE} (
        request_id STRING NOT NULL,
        timestamp TIMESTAMP NOT NULL,
        endpoint_name STRING NOT NULL,
        model_id STRING,
        model_version STRING,
        input_fare DOUBLE,
        input_age DOUBLE,
        input_pclass INT,
        input_sibsp INT,
        input_parch INT,
        input_sex STRING,
        prediction INT,
        prediction_probability DOUBLE,
        actual INT,
        response_time_ms DOUBLE,
        status_code INT
    )
    USING DELTA
    COMMENT 'Inference logs for Titanic model serving endpoint'
""")

existing_columns = {field.name for field in spark.table(INFERENCE_LOG_TABLE).schema.fields}
new_columns = []

if "model_id" not in existing_columns:
    new_columns.append("model_id STRING")
if "prediction_probability" not in existing_columns:
    new_columns.append("prediction_probability DOUBLE")
if "actual" not in existing_columns:
    new_columns.append("actual INT")

if new_columns:
    spark.sql(f"ALTER TABLE {INFERENCE_LOG_TABLE} ADD COLUMNS ({', '.join(new_columns)})")
    print(f"Added missing columns: {', '.join(new_columns)}")

print(f"Inference log table created/verified: {INFERENCE_LOG_TABLE}")
print(f"Current columns: {', '.join(spark.table(INFERENCE_LOG_TABLE).columns)}")

In [0]:
# %sql
# TRUNCATE TABLE ml_catalog.titanic_schema.inference_logs;
# TRUNCATE TABLE ml_catalog.titanic_schema.inference_logs_profile_ready;

In [0]:
import uuid
import json
import requests
import time
from datetime import datetime
import pandas as pd
import numpy as np

def log_inference_request(
    url,
    headers,
    payload_json,
    endpoint_name="example-titanic-serving-stg-v2",
    model_version="3",
    actual_label=None,
    model_id=None
):
    """
    Make an inference request and log the payload and response to Delta table.
    
    Args:
        url: Model serving endpoint URL
        headers: Request headers with authorization token
        payload_json: JSON string of the input data
        endpoint_name: Name of the serving endpoint
        model_version: Version of the model being served
        actual_label: Ground-truth label for monitoring, if available
        model_id: Stable identifier for the served model
    
    Returns:
        dict: Response from the endpoint
    """
    # Generate unique request ID
    request_id = str(uuid.uuid4())
    timestamp = datetime.now()
    derived_model_id = model_id or f"{endpoint_name}_v{model_version}"
    
    # Parse input payload
    payload_dict = json.loads(payload_json)
    input_data = payload_dict["dataframe_split"]["data"][0]
    
    # Make the request and measure response time
    start_time = time.time()
    response = None
    response_payload = None
    prediction = None
    prediction_probability = None
    
    try:
        response = requests.post(url=url, headers=headers, data=payload_json)
        response_time_ms = (time.time() - start_time) * 1000
        status_code = response.status_code
        
        if response.status_code == 200:
            response_payload = response.json()
            predictions = response_payload.get("predictions", [])
            
            if predictions:
                first_prediction = predictions[0]
                if isinstance(first_prediction, dict):
                    prediction = first_prediction.get("prediction")
                    if prediction is None:
                        prediction = first_prediction.get("predicted_label", first_prediction.get("class"))
                    prediction_probability = first_prediction.get("probability")
                    if prediction_probability is None:
                        prediction_probability = first_prediction.get("predicted_probability", first_prediction.get("score"))
                else:
                    prediction = first_prediction
            
            if prediction_probability is None:
                probabilities = response_payload.get("probabilities") or response_payload.get("prediction_probabilities")
                if isinstance(probabilities, list) and probabilities:
                    first_probability = probabilities[0]
                    if isinstance(first_probability, list) and first_probability:
                        prediction_probability = float(max(first_probability))
                    elif isinstance(first_probability, (int, float)):
                        prediction_probability = float(first_probability)
                elif isinstance(response_payload.get("prediction_probability"), (int, float)):
                    prediction_probability = float(response_payload["prediction_probability"])

            if prediction is not None:
                prediction = int(prediction)
        else:
            print(f"Request failed with status {response.status_code}: {response.text}")
    
    except Exception as e:
        response_time_ms = (time.time() - start_time) * 1000
        status_code = 500
        print(f"Error during request: {e}")
    
    # Create log entry
    log_entry = {
        "request_id": request_id,
        "timestamp": timestamp,
        "endpoint_name": endpoint_name,
        "model_id": derived_model_id,
        "model_version": model_version,
        "input_fare": float(input_data[0]),
        "input_age": float(input_data[1]),
        "input_pclass": int(input_data[2]),
        "input_sibsp": int(input_data[3]),
        "input_parch": int(input_data[4]),
        "input_sex": str(input_data[5]),
        "prediction": prediction,
        "prediction_probability": prediction_probability,
        "actual": int(actual_label) if actual_label is not None and not pd.isna(actual_label) else None,
        "response_time_ms": response_time_ms,
        "status_code": status_code
    }
    
    # Convert to DataFrame and append to Delta table
    log_df = spark.createDataFrame([log_entry], schema=schema)
    log_df.write.option("mergeSchema", "true").format("delta").mode("append").saveAsTable(INFERENCE_LOG_TABLE)
    
    print(
        f"✓ Logged request {request_id} - Model ID: {derived_model_id}, "
        f"Prediction: {prediction}, Probability: {prediction_probability}, Response time: {response_time_ms:.2f}ms"
    )
    
    return response_payload if response and response.status_code == 200 else None

print("Function 'log_inference_request' ready to use")

In [0]:
# Test the logging function with a single request
url = 'https://dbc-b9c143cb-aa13.cloud.databricks.com/serving-endpoints/example-titanic-serving-stg-v2/invocations'
token = dbutils.secrets.get(scope='my-secrets', key='databricks-token')
headers = {'Authorization': f'Bearer {token}', 'Content-Type': 'application/json'}

# Use a test sample and keep the matching label
sample_idx = X.index[0]
test_payload = {"dataframe_split": X.loc[[sample_idx]].to_dict(orient="split")}
test_payload_json = json.dumps(test_payload, allow_nan=True)

# Make request and log
result = log_inference_request(
    url=url,
    headers=headers,
    payload_json=test_payload_json,
    endpoint_name="example-titanic-serving-stg-v2",
    model_version="3",
    actual_label=int(y.loc[sample_idx])
)

print(f"\nResult: {result}")

In [0]:
# Simulate continuous inference requests (like a live production scenario)
# This will make 20 requests with random samples from the training data

import time
import random
import json

print("Starting live inference simulation...\n")

available_indices = list(X.index)
num_requests = 20
for i in range(num_requests):
    # Get a random sample from training data
    random_idx = random.choice(available_indices)
    sample_payload = {"dataframe_split": X.loc[[random_idx]].to_dict(orient="split")}
    sample_payload_json = json.dumps(sample_payload, allow_nan=True)
    
    # Log the inference
    log_inference_request(
        url=url,
        headers=headers,
        payload_json=sample_payload_json,
        endpoint_name="example-titanic-serving-stg-v2",
        model_version="3",
        actual_label=int(y.loc[random_idx])
    )
    
    # Small delay between requests (adjust as needed)
    time.sleep(0.5)

print(f"\n✓ Completed {num_requests} inference requests with logging")

In [0]:
# Query the inference logs from Delta table
inference_logs = spark.table(INFERENCE_LOG_TABLE)

# Show recent logs
print("Recent inference logs:")
display(
    inference_logs
    .orderBy("timestamp", ascending=False)
    .select(
        "request_id",
        "timestamp",
        "endpoint_name",
        "model_id",
        "model_version",
        "prediction",
        "prediction_probability",
        "actual",
        "response_time_ms",
        "status_code"
    )
    .limit(10)
)

# Summary statistics
print("\nInference Summary:")
summary_df = inference_logs.selectExpr(
    "count(*) as total_requests",
    "avg(response_time_ms) as avg_response_time_ms",
    "max(response_time_ms) as max_response_time_ms",
    "min(response_time_ms) as min_response_time_ms",
    "avg(prediction_probability) as avg_prediction_probability",
    "sum(case when actual is not null then 1 else 0 end) as rows_with_actual",
    "sum(case when status_code = 200 then 1 else 0 end) as successful_requests",
    "sum(case when status_code != 200 then 1 else 0 end) as failed_requests"
)

display(summary_df)

In [0]:
# Analyze prediction distribution
prediction_dist = spark.sql(f"""
    SELECT 
        prediction,
        COUNT(*) as count,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) as percentage
    FROM {INFERENCE_LOG_TABLE}
    GROUP BY prediction
    ORDER BY prediction
""")

print("Prediction Distribution:")
display(prediction_dist)

# Time-based analysis
time_analysis = spark.sql(f"""
    SELECT 
        DATE_TRUNC('minute', timestamp) as minute,
        COUNT(*) as requests_per_minute,
        AVG(response_time_ms) as avg_response_time,
        AVG(prediction) as avg_prediction
    FROM {INFERENCE_LOG_TABLE}
    GROUP BY DATE_TRUNC('minute', timestamp)
    ORDER BY minute DESC
    LIMIT 10
""")

print("\nRequests Over Time:")
display(time_analysis)

In [0]:
def send_request(request_id):
    """Send a single request and record metrics"""
    start_time = time.time()
    try:
        response = requests.post(url, headers=headers, data=payload_json, timeout=30)
        latency = time.time() - start_time
        
        return {
            'request_id': request_id,
            'status_code': response.status_code,
            'latency': latency,
            'success': response.status_code == 200,
            'response': response.json() if response.status_code == 200 else None,
            'error': None
        }
    except Exception as e:
        latency = time.time() - start_time
        return {
            'request_id': request_id,
            'status_code': None,
            'latency': latency,
            'success': False,
            'response': None,
            'error': str(e)
        }

def load_test(num_requests=100, num_workers=10):
    """Run load test with concurrent requests"""
    print(f"Starting load test: {num_requests} requests with {num_workers} concurrent workers...\n")
    
    results = []
    start_time = time.time()
    
    with ThreadPoolExecutor(max_workers=num_workers) as executor:
        futures = [executor.submit(send_request, i) for i in range(num_requests)]
        
        for future in as_completed(futures):
            results.append(future.result())
            if len(results) % 1000 == 0:
                print(f"Completed {len(results)}/{num_requests} requests")
    
    total_time = time.time() - start_time
    
    # Calculate metrics
    latencies = [r['latency'] for r in results]
    successes = [r for r in results if r['success']]
    failures = [r for r in results if not r['success']]
    
    print("\n" + "="*50)
    print("LOAD TEST RESULTS")
    print("="*50)
    print(f"Total requests: {num_requests}")
    print(f"Concurrent workers: {num_workers}")
    print(f"Total time: {total_time:.2f} seconds")
    print(f"Throughput: {num_requests/total_time:.2f} requests/second")
    print(f"\nSuccess rate: {len(successes)/num_requests*100:.2f}% ({len(successes)}/{num_requests})")
    print(f"Failed requests: {len(failures)}")
    print(f"\nLatency statistics (seconds):")
    print(f"  Min: {np.min(latencies):.3f}")
    print(f"  Max: {np.max(latencies):.3f}")
    print(f"  Mean: {np.mean(latencies):.3f}")
    print(f"  Median: {np.median(latencies):.3f}")
    print(f"  P95: {np.percentile(latencies, 95):.3f}")
    print(f"  P99: {np.percentile(latencies, 99):.3f}")
    
    if failures:
        print(f"\nError samples:")
        for f in failures[:5]:
            print(f"  Request {f['request_id']}: {f['error'] or f'HTTP {f['status_code']}'}")
    
    return results

In [0]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Assume that your endpoint receives at max 25 concurrent requests at a time and p99 latency requirement is 250ms. Can your model meet these requirements?
results = load_test(num_requests=5000, num_workers=25)